# Sociodemographic, Clinical, Pregnancy, and Neonatal Variables for Pregnant Women With and Without Epilepsy in Alagoas, Brazil, 2021–2022 Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR\^2 dataset on pregnancy and neonatal variables in women with and without epilepsy using the `mlcroissant` library.

### Dataset Source
The dataset Croissant schema is available from:
https://sen.science/doi/10.71728/senscience.tw2e-pz0d/fair2.json

This dataset contains sociodemographic, clinical, obstetric, and neonatal data collected from pregnant women with and without epilepsy between 2021 and 2022 at multiple healthcare centers in Alagoas, Brazil.

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and make a first check of the available information using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.tw2e-pz0d/fair2.json'

# Initialize mlcroissant Dataset
dataset = mlc.Dataset(croissant_url)

# Extract metadata (as dict to pretty-print)
metadata_json = dataset.metadata.to_json()
print(f"Dataset Name: {metadata_json['name']}")
print(f"Description: {metadata_json['description']}")
print(f"Identifier: {metadata_json['identifier']}")
print(f"Temporal Coverage: {metadata_json.get('temporalCoverage', 'N/A')}")
print(f"Spatial Coverage: {metadata_json.get('spatialCoverage', 'N/A')}")

## 2. Data Overview
List available record sets and, for each, their fields and columns. All entities are referenced strictly by their `@id` as specified in the Croissant schema.

In [ ]:
# Show all available record sets by @id
record_sets = list(dataset.record_sets)
print("Record Set @id List:")
for rs in record_sets:
    print(f"  - {rs['@id']}: {rs['name']}")

# For each record set, list its fields and columns by @id
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']} (name: {rs['name']})")
    print("  Fields:")
    for field in rs['field']:
        # Try to get field details or just id
        field_obj = dataset.field_by_id(field) if hasattr(dataset, 'field_by_id') else None
        if field_obj is not None:
            print(f"    - {field_obj['@id']}: {field_obj['name']}")
        else:
            print(f"    - {field}")
    print("  Columns:")
    if 'column' in rs:
        for column in rs['column']:
            print(f"    - {column}")

## 3. Data Extraction
Select a record set of interest by its `@id` and extract the records into a DataFrame for further analysis. All references are made via the entity `@id`s.

**Note:** See section 2 for valid `@id` values.

In [ ]:
# --- Select record sets to load ---
# Choose real @id(s) from record_sets found above. Below, replace with actual IDs as appropriate.
selected_record_set_ids = [rs['@id'] for rs in record_sets]
# Example: selected_record_set_ids = ['cr:RecordSet0', 'cr:RecordSet1']

# Load all record sets to a dictionary of DataFrames
dataframes = {}
for record_set_id in selected_record_set_ids:
    try:
        # Records is an iterator of dicts
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} rows for record set {record_set_id}")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")

# Show columns for the first record set
if selected_record_set_ids:
    example_rs_id = selected_record_set_ids[0]
    print(f"Available columns in record set {example_rs_id}:")
    print(dataframes[example_rs_id].columns.tolist())
    # Show first few rows
    display(dataframes[example_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Demonstrate basic data analysis: filtering, normalization, outlier removal, or grouping by key variables. All fields are referenced using their `@id` as per FAIR \^2 Croissant schema.

_Please update the variable placeholders with actual `@id` values matching your field of interest._

In [ ]:
# --- Example EDA: Analyze Age and Group by Epilepsy Status ---
# Update these to the correct @id for your numeric and group field.
rs_id = example_rs_id  # analyzed record set
df = dataframes[rs_id]

# Replace with the actual field @id or DataFrame column name for 'age' and 'epilepsy_status'.
numeric_field_id = None
group_field_id = None

# Try to guess one (e.g., often 'age' or similar in columns)
for col in df.columns:
    if 'age' in col.lower():
        numeric_field_id = col
    if 'epilepsy' in col.lower() or 'group' in col.lower():
        group_field_id = col

print(f"Numeric field selected for EDA: {numeric_field_id}")
print(f"Grouping field selected for EDA: {group_field_id}")

# Check that numeric field is present
if numeric_field_id in df.columns:
    # Convert to numeric if necessary
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    # Example threshold for age
    threshold = 18
    filtered_df = df[df[numeric_field_id] >= threshold].copy()
    print(f"Filtered records with {numeric_field_id} >= {threshold} (rows: {len(filtered_df)}):")
    display(filtered_df[[numeric_field_id]].head())

    # Normalize age
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
    print(f"First records with normalized {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by epilepsy status if available
    if group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped {numeric_field_id} mean by {group_field_id}:")
        print(grouped_df)
else:
    print(f"Numeric field '{numeric_field_id}' not found in DataFrame.")

## 5. Visualization
Visualize numeric data distributions or relationships between key variables using common Python plotting libraries. Adjust the field `@id`s as needed.

_This example visualizes the age distribution and average age by epilepsy status if group variable is available._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("No numeric field found or selected for visualization.")

## 6. Conclusion
This notebook demonstrated how to:
- Load and explore a FAIR\^2 published clinical cohort dataset using `mlcroissant`
- Reference all record sets, fields, and columns strictly by their `@id` to ensure schema traceability
- Perform and visualize simple exploratory data analysis

_For further analysis, consult the Croissant schema and documentation for variable definitions, or use additional columns and record sets as needed._